# 07 - Random Survival Forest

RSF xử lý quan hệ phi tuyến và tương tác tốt hơn Cox PH.

Notebook dùng cùng **project-level split** với Cox để so sánh công bằng.

In [1]:
from pathlib import Path
import sys, yaml, pandas as pd, numpy as np

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

with open(ROOT / "configs" / "experiment.yaml", "r", encoding="utf-8") as f:
    CFG = yaml.safe_load(f)

print("ROOT =", ROOT)
print("random_seed =", CFG["random_seed"])

ROOT = d:\VNUK\Eureka 2026\Eureka_2026
random_seed = 42


In [2]:
from sklearn.model_selection import GroupShuffleSplit

MODE = "day0"  # đổi thành "day7" nếu muốn chạy landmark model
data = pd.read_parquet(ROOT / "data" / "processed" / f"model_{MODE}.parquet")

groups = data["project"].astype(str)
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=CFG["random_seed"])
train_idx, temp_idx = next(gss1.split(data, groups=groups))

train = data.iloc[train_idx].copy()
temp = data.iloc[temp_idx].copy()

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=CFG["random_seed"] + 1)
val_rel, test_rel = next(gss2.split(temp, groups=temp["project"].astype(str)))

val = temp.iloc[val_rel].copy()
test = temp.iloc[test_rel].copy()

print("Train projects:", sorted(train["project"].unique()))
print("Validation projects:", sorted(val["project"].unique()))
print("Test projects:", sorted(test["project"].unique()))

assert set(train["project"]).isdisjoint(set(val["project"]))
assert set(train["project"]).isdisjoint(set(test["project"]))
assert set(val["project"]).isdisjoint(set(test["project"]))

Train projects: ['FLEX', 'HIVE', 'JRACLOUD', 'JRASERVER', 'MC', 'MCPE', 'MDEV', 'SERVER']
Validation projects: ['CONFSERVER', 'SAK']
Test projects: ['OSSRH', 'QTBUG']


In [3]:
from src.features import feature_spec, sample_training_rows
from src.models import fit_rsf, save_artifact
from src.evaluation import evaluate_survival_model

strict = bool(CFG["features"]["strict_no_leakage"])
landmark = int(CFG["features"]["landmark_days"])
numeric_cols, categorical_cols = feature_spec(
    MODE,
    strict_no_leakage=strict,
    landmark_days=landmark,
)

max_rows = 50000
train_fit = sample_training_rows(
    train,
    max_rows=max_rows,
    random_state=CFG["random_seed"],
)

rsf_cfg = CFG["models"]["rsf"].copy()
rsf_cfg["random_state"] = CFG["random_seed"]
rsf_cfg["n_estimators"] = 50
rsf_cfg["n_jobs"] = 1
rsf_cfg["max_depth"] = 15

print("Train rows used:", len(train_fit), "/", len(train))
print(rsf_cfg)

Train rows used: 50000 / 576250
{'n_estimators': 50, 'max_depth': 15, 'min_samples_split': 20, 'min_samples_leaf': 10, 'max_features': 'sqrt', 'n_jobs': 1, 'random_state': 42}


In [4]:
rsf_artifact = fit_rsf(
    train_df=train_fit,
    numeric_cols=numeric_cols,
    categorical_cols=categorical_cols,
    **rsf_cfg,
)

val_sub = val.sample(n=min(5000, len(val)), random_state=42)
test_sub = test.sample(n=min(5000, len(test)), random_state=42)
metrics_val = evaluate_survival_model(
    rsf_artifact,
    train_df=train_fit,
    test_df=val_sub,
    horizons_days=CFG["evaluation"]["horizons_days"],
)
metrics_test = evaluate_survival_model(
    rsf_artifact,
    train_df=train_fit,
    test_df=test_sub,
    horizons_days=CFG["evaluation"]["horizons_days"],
)

pd.DataFrame([
    {"split": "validation", **metrics_val},
    {"split": "test", **metrics_test},
])

,split,n_test,events_test,harrell_c,ipcw_c,auc_30,auc_60,auc_90,mean_dynamic_auc,ibs
0,validation,5000,4376,0.616681,0.606134,0.672702,0.672223,0.678287,0.673044,0.195977
1,test,5000,4329,0.373740,0.377909,0.329809,0.344749,0.354557,0.331437,0.165449


In [5]:
model_dir = ROOT / "results" / "models"
table_dir = ROOT / "results" / "tables"
model_dir.mkdir(parents=True, exist_ok=True)
table_dir.mkdir(parents=True, exist_ok=True)

save_artifact(rsf_artifact, model_dir / f"rsf_{MODE}.joblib")

metrics_df = pd.DataFrame([
    {"model": "RSF", "mode": MODE, "split": "validation", **metrics_val},
    {"model": "RSF", "mode": MODE, "split": "test", **metrics_test},
])
metrics_df.to_csv(table_dir / f"rsf_metrics_{MODE}.csv", index=False)

metrics_df

,model,mode,split,n_test,events_test,harrell_c,ipcw_c,auc_30,auc_60,auc_90,mean_dynamic_auc,ibs
0,RSF,day0,validation,5000,4376,0.616681,0.606134,0.672702,0.672223,0.678287,0.673044,0.195977
1,RSF,day0,test,5000,4329,0.373740,0.377909,0.329809,0.344749,0.354557,0.331437,0.165449


In [6]:
# So sánh Cox vs RSF nếu notebook 06 đã chạy.
cox_file = table_dir / f"cox_metrics_{MODE}.csv"
rsf_file = table_dir / f"rsf_metrics_{MODE}.csv"

parts = []
if cox_file.exists():
    parts.append(pd.read_csv(cox_file))
parts.append(pd.read_csv(rsf_file))

comparison = pd.concat(parts, ignore_index=True)
comparison.to_csv(table_dir / f"model_comparison_{MODE}.csv", index=False)
comparison

,model,mode,split,n_test,events_test,harrell_c,ipcw_c,auc_30,auc_60,auc_90,mean_dynamic_auc,ibs
0,CoxPH,day0,validation,87261,76376,0.595495,0.588819,0.635460,0.640160,0.643017,0.636529,0.198562
1,CoxPH,day0,test,172132,149713,0.374037,0.378573,0.309602,0.338403,0.352223,0.312770,NaN
2,RSF,day0,validation,5000,4376,0.616681,0.606134,0.672702,0.672223,0.678287,0.673044,0.195977
3,RSF,day0,test,5000,4329,0.373740,0.377909,0.329809,0.344749,0.354557,0.331437,0.165449


Nếu RSF mất quá nhiều thời gian/RAM:
- giảm `models.max_train_rows`;
- giảm `n_estimators` để smoke-test;
- khi pipeline ổn mới tăng lại để chạy kết quả cuối.